# 保存済み歩行モデルを表示する — RoboQuest2026

**再学習は不要です。3つのセルを上から実行してください。**

1. セットアップ
2. Google Driveに接続
3. フォルダ名とZIP名を指定して表示

Driveの元ファイルは変更しません。正規化データがない場合は参考プレビューと明示します。


In [ ]:
#@title 🔧 セットアップ（最初に一度だけ実行してください）

import subprocess, sys, os, zipfile

print(f'Python {sys.version_info.major}.{sys.version_info.minor} で実行中')

subprocess.run(
    'command -v ffmpeg >/dev/null || (apt-get update -qq && apt-get install -y -q ffmpeg)',
    shell=True, check=False)

# ローカル検証済みのコード一式を使う場合は、先にこのZIPをColabにアップロード。
_bundle = '/content/walk_colab_bundle.zip'
if os.path.isfile(_bundle):
    with zipfile.ZipFile(_bundle) as archive:
        for member in archive.namelist():
            destination = os.path.realpath(os.path.join('/content', member))
            if not destination.startswith('/content/RoboQuest2026/'):
                raise ValueError('想定外のファイル名を含むZIPです。')
        archive.extractall('/content')
    print('ローカルと同じコード・モデルを読み込みました。')
elif not os.path.exists('/content/RoboQuest2026/.git'):
    print('リポジトリをダウンロード中...')
    subprocess.run(['git', 'clone', '-q',
        'https://github.com/SingularityBattleQuest/RoboQuest2026.git',
        '/content/RoboQuest2026'], check=True)
else:
    print('リポジトリを最新化中...')
    subprocess.run(['git', '-C', '/content/RoboQuest2026', 'pull', 'origin', 'main', '-q'],
                   check=False)

os.chdir('/content/RoboQuest2026')
if '/content/RoboQuest2026' not in sys.path:
    sys.path.insert(0, '/content/RoboQuest2026')

print('ライブラリをインストール中...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    '-r', 'requirements.txt',
], check=True)

print('ブラウザビューアー (mjswan) をインストール中...')
_mjswan_flags = ['--ignore-requires-python'] if sys.version_info >= (3, 13) else []
MJSWAN_AVAILABLE = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', *_mjswan_flags, '-c', 'requirements-training.txt', 'mjswan==0.8.2'],
).returncode == 0
if not MJSWAN_AVAILABLE:
    print('⚠ mjswan のインストールに失敗しました。ビューアーのセルだけが使えません。')
    print('  学習・数値評価のセルはそのまま実行できます。講師に連絡してください。')

print('Go2 ロボットモデルをダウンロード中...')
subprocess.run([sys.executable, 'scripts/download_models.py'], check=True)

print('\n✅ セットアップ完了！次のセルへ進んでください。')


In [ ]:
#@title Google Driveに接続
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
#@title 📂 保存済み歩行モデルを選んで表示（再学習不要）
#@markdown Google Drive の RoboQuest2026 内のフォルダ名を指定してください。
saved_team_name = "test" #@param {type:"string"}
#@markdown 実際のZIP名を指定（walkmodel.zipの場合は変更してください）。
saved_model_name = "walk_model.zip" #@param {type:"string"}
#@markdown 正規化ファイルが別名ならフルパスを指定。通常は空欄。
saved_vecnorm_path = "" #@param {type:"string"}

from pathlib import Path
root = Path('/content/drive/MyDrive/RoboQuest2026')
folder = (root / saved_team_name).resolve()
if not folder.is_relative_to(root.resolve()):
    raise ValueError('RoboQuest2026内のフォルダ名を指定してください')
if not folder.is_dir():
    available = ', '.join(p.name for p in root.iterdir() if p.is_dir()) if root.exists() else '(Drive未接続)'
    raise FileNotFoundError(f'フォルダがありません。選べるフォルダ: {available}')
model_path = folder / saved_model_name
# Accept the common spelling without an underscore as well.
if not model_path.exists() and saved_model_name == 'walk_model.zip':
    model_path = folder / 'walkmodel.zip'
if not model_path.is_file():
    raise FileNotFoundError(f'ZIPがありません。候補: {[p.name for p in folder.glob("*.zip")]}')
print(f'読み込むモデル: {model_path}')

"""Serve a built, single-threaded mjswan viewer inside Google Colab.

This file can also be pasted into a new Colab cell after building the viewer.
It does not rebuild the viewer or modify the trained policy.
"""
from functools import partial
from html import escape
from http.server import SimpleHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path
import threading
from urllib.request import urlopen


class ViewerHandler(SimpleHTTPRequestHandler):
    extensions_map = {
        **SimpleHTTPRequestHandler.extensions_map,
        ".js": "application/javascript",
        ".mjs": "application/javascript",
        ".wasm": "application/wasm",
    }

    def end_headers(self):
        # Each rebuild may replace config and policy files at the same URL.
        self.send_header("Cache-Control", "no-store")
        super().end_headers()

    def log_message(self, format, *args):
        pass


def start_viewer_server(directory):
    directory = Path(directory).resolve()
    for required in ("index.html", "assets/config.json"):
        if not (directory / required).is_file():
            raise FileNotFoundError(
                f"ビューアーのファイルがありません: {directory / required}\n"
                "先にビューアーのビルドを完了してください。再学習は不要です。"
            )
    # Bind before displaying the iframe. Port 0 allocates a free port atomically.
    # A threaded server avoids one idle browser connection blocking all assets.
    server = ThreadingHTTPServer(
        ("127.0.0.1", 0), partial(ViewerHandler, directory=str(directory))
    )
    threading.Thread(target=server.serve_forever, daemon=True).start()
    try:
        with urlopen(f"http://127.0.0.1:{server.server_port}/index.html", timeout=10) as response:
            if response.status != 200:
                raise RuntimeError("ビューアーのHTTP起動確認に失敗しました。")
    except Exception:
        server.shutdown()
        server.server_close()
        raise
    return server


def launch_colab_viewer(directory="/tmp/rq_walk_dist", height=620):
    from google.colab import output
    from IPython.display import HTML, display

    server = start_viewer_server(directory)
    try:
        # Resolve explicitly: errors reach the cell instead of leaving an empty
        # async Javascript output. Use the authenticated Colab proxy, not a tunnel.
        url = output.eval_js(f"google.colab.kernel.proxyPort({server.server_port})")
        if not isinstance(url, str) or not url.startswith("https://"):
            raise RuntimeError("ColabのビューアーURLを取得できませんでした。")
        display(HTML(
            '<p>3Dビューアーを読み込みます。初回はWASMの転送に時間がかかります。</p>'
            f'<iframe src="{escape(url, quote=True)}" width="100%" '
            f'height="{int(height)}" style="border:0" '
            'title="RoboQuest 学習済みモデル" allow="autoplay; fullscreen"></iframe>'
        ))
    except Exception:
        server.shutdown()
        server.server_close()
        raise
    print("✅ 表示サーバー起動・Colab接続確認済み（3D描画完了の確認は画面で行ってください）")
    return server


"""Export a selected saved Walk checkpoint without retraining it."""
from pathlib import Path
import tempfile


def preview_saved_walk(model_path, vecnorm_path=None):
    from stable_baselines3 import PPO
    from scripts.export_for_web import export_normalized_policy_onnx

    model_path = Path(model_path)
    if not model_path.is_file():
        raise FileNotFoundError(f"モデルが見つかりません: {model_path}")
    model = PPO.load(str(model_path), device="cpu")
    if model.observation_space.shape != (45,) or model.action_space.shape != (12,):
        raise ValueError(
            f"このモデルは現在のWalkビューアーに対応しません。"
            f"観測={model.observation_space.shape}, 行動={model.action_space.shape}。"
            "必要な形は観測45・行動12です。"
        )
    # Separate each export so a previous team's ONNX/stats can never be reused.
    preview_dir = Path(tempfile.mkdtemp(prefix="rq_saved_walk_"))
    if vecnorm_path:
        stats = Path(vecnorm_path)
        if not stats.is_file():
            raise FileNotFoundError(f"指定した正規化データがありません: {stats}")
    else:
        candidates = [
            model_path.with_name(model_path.stem + "_vecnorm.pkl"),
            model_path.with_name("walk_vecnorm.pkl"),
        ]
        stats = next((p for p in candidates if p.is_file()), preview_dir / "missing.pkl")
    has_stats = stats.is_file()
    if has_stats:
        print(f"✅ 正規化データ: {stats.name}")
    else:
        print("⚠ 正規化データがありません。ZIPのみでプレビューします。")
        print("学習時と動きが異なったり、転倒したりする可能性があります。")
        print("この表示だけで学習の成功・失敗を判定しないでください。")
    onnx_path = preview_dir / "walk_policy_normalized.onnx"
    export_normalized_policy_onnx(model_path, stats, onnx_path)
    return onnx_path, has_stats


from scripts.build_mjswan_viewer import build_walk
onnx_path, has_stats = preview_saved_walk(model_path, saved_vecnorm_path or None)
app = build_walk(onnx_path, onnx_path.parent / 'viewer')
if 'saved_walk_viewer_server' in globals():
    saved_walk_viewer_server.shutdown()
    saved_walk_viewer_server.server_close()
saved_walk_viewer_server = launch_colab_viewer(onnx_path.parent / 'viewer', height=620)
if not has_stats:
    from IPython.display import HTML, display
    display(HTML('<p style="color:#b45309"><b>参考プレビュー：正規化データなし。学習時の動作再現ではありません。</b></p>'))
